# 05b. Qwen3-0.6B with LoRA: training on Kaggle

Training run for the LLM candidate, executed on Kaggle (two Tesla T4 GPUs) because the model does not fit the laptop's 6 GB GPU. Qwen3-0.6B gets a two-class classification head and LoRA adapters on its attention layers, so only 4.6 million of its 600 million parameters are trained. It reads the same 202,530 sentences as DistilBERT and ModernBERT, for one epoch. Only the adapter and the training log are saved. Validation is scored in a separate run (`05c_qwen_lora_score_kaggle.ipynb`), because training and scoring together would not fit in Kaggle's 12 hour limit.

In [1]:
!pip install -q --no-deps --force-reinstall git+https://github.com/sap6011/bank-guard.git
!pip install -q --no-deps peft
!pip uninstall -y -q torchao

import importlib.metadata as md

for package in ["torch", "transformers", "peft", "accelerate"]:
    print(package, md.version(package))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
torch 2.11.0+cu130
transformers 5.20.2
peft 0.21.2
accelerate 1.15.0


The `bankguard` package is installed from GitHub, so the features and sentences are built by the same code as on the laptop. PEFT is installed without its dependencies so that Kaggle's torch and transformers stay unchanged. torchao is uninstalled because Kaggle ships version 0.10.0, older than PEFT requires, and PEFT stops with an error instead of skipping it; the model is not quantised, so torchao is not needed.

In [2]:
import hashlib
from pathlib import Path

import numpy as np
import torch

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.features import FEATURES, build_features, fit_currency_median
from bankguard.text import to_text

DATA = Path("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml")

tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]

X = build_features(tx, accounts, fit_currency_median(train))
X_train, y_train = X.loc[train.index, FEATURES], train["is_laundering"].to_numpy()
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()
del tx, X

# Same 200,000 normal rows as DistilBERT and ModernBERT, so the hashes can be checked.
rng = np.random.default_rng(42)
pos = np.flatnonzero(y_train == 1)
neg = rng.choice(np.flatnonzero(y_train == 0), size=200_000, replace=False)
sample = np.sort(np.concatenate([pos, neg]))
train_text = to_text(X_train.iloc[sample]).tolist()
val_text = to_text(X_val).tolist()
print("train text:", hashlib.md5("\n".join(train_text).encode()).hexdigest())
print("val text:  ", hashlib.md5("\n".join(val_text).encode()).hexdigest())

# The LLM trains on 50,000 of those 200,000 normal rows, plus every laundering row.
small_neg = np.random.default_rng(7).choice(neg, size=50_000, replace=False)
llm_sample = np.sort(np.concatenate([pos, small_neg]))
llm_text = to_text(X_train.iloc[llm_sample]).tolist()
llm_labels = y_train[llm_sample]
print(f"LLM training rows: {len(llm_text):,}, {llm_labels.mean():.2%} laundering")

train text: 6b7502e690497d6d06dbd6ae12a15e96
val text:   ed4defe73ab6fe9ee28e95ea9e7f12b3
LLM training rows: 52,530, 4.82% laundering


The training and validation sentences give the same MD5 hashes as for DistilBERT and ModernBERT (train 6b7502e6, validation ed4defe7), so all three text models read identical inputs. The last lines build a smaller training set of 52,530 rows (every laundering row plus 50,000 normal rows), which was planned when training was expected to be slow. It was used only for the speed probe below. Once the probe showed training was fast, the full run used all 202,530 sentences instead.

In [3]:
import time

from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL = "Qwen/Qwen3-0.6B"
MAX_LEN = 192
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lengths = [len(ids) for ids in tokenizer(llm_text[:2000])["input_ids"]]
print(f"tokens per sentence: mean {np.mean(lengths):.0f}, max {max(lengths)}")


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return {"text": self.texts[i], "label": int(self.labels[i])}


def collate(batch):
    enc = tokenizer(
        [b["text"] for b in batch],
        truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt",
        return_token_type_ids=False,
    )
    enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc


def make_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=2, dtype=torch.float32
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    lora = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    return get_peft_model(model, lora)


train_ds = TextDataset(llm_text, llm_labels)
model = make_model()
model.print_trainable_parameters()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 4,589,568 || all params: 600,641,536 || trainable%: 0.7641


Qwen3-0.6B is loaded in full precision with a new two-class head (`score`), since the T4 does not support bf16. The load report is expected: the language model head is dropped and the classification head is newly created. LoRA adds small trainable matrices of rank 16 (alpha 32, dropout 0.05) to the query, key, value and output projections, leaving 99.24% of the weights frozen. Because a decoder model classifies from its last real token, the padding token is set so the model can find that token in padded batches.

In [4]:
probe_args = TrainingArguments(
    output_dir="/kaggle/temp/qwen-probe",
    max_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=25,
    save_strategy="no",
    eval_strategy="no",
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

torch.cuda.reset_peak_memory_stats()
probe = Trainer(model=model, args=probe_args, train_dataset=train_ds, data_collator=collate)
out = probe.train()
steps_per_s = out.metrics["train_steps_per_second"]
print(f"training: {steps_per_s:.2f} steps/s, about {len(train_ds) // 32 / steps_per_s / 60:.0f} min for 1 epoch")

start = time.time()
probe.predict(TextDataset(val_text[:4000], y_val[:4000]))
rows_per_s = 4000 / (time.time() - start)
print(f"scoring: {rows_per_s:.0f} rows/s, about {len(val_text) / rows_per_s / 60:.0f} min for all validation")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
25,0.364449
50,0.338283


training: 0.49 steps/s, about 56 min for 1 epoch


scoring: 41 rows/s, about 394 min for all validation
peak GPU memory: 9.67 GB


A 50 step interactive probe checked the setup before the long background run: 0.49 steps per second (about 56 minutes per epoch on 52,530 rows), scoring at 41 rows per second (about 6.5 hours for all of validation), and peak GPU memory of 9.67 GB of 16. Training was much faster than estimated, which made it possible to train on the full 202,530 sentences.

In [5]:
import gc
import json

del probe
gc.collect()
torch.cuda.empty_cache()

model = make_model()  # fresh weights: the probe changed the LoRA matrices

args = TrainingArguments(
    output_dir="/kaggle/temp/qwen-lora",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=100,
    fp16=True,
    logging_steps=50,
    save_strategy="no",
    eval_strategy="no",
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

train_ds = TextDataset(train_text, y_train[sample])  # the same 202,530 sentences as the BERT models
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
start = time.time()
result = trainer.train()
train_minutes = (time.time() - start) / 60

model.save_pretrained("/kaggle/working/qwen_lora_adapter")
tokenizer.save_pretrained("/kaggle/working/qwen_lora_adapter")
with open("/kaggle/working/qwen_train_log.json", "w") as f:
    json.dump({"train_minutes": train_minutes, "log_history": trainer.state.log_history}, f, indent=2)
print(f"training time: {train_minutes:.1f} min, adapter saved")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,3.456410
100,0.168195
150,0.154541
200,0.191521
250,0.192048
300,0.140033
350,0.139149
400,0.150478
450,0.153225
500,0.149263


training time: 223.9 min, adapter saved


Training ran for one epoch over 202,530 sentences (6,330 steps of 32 sentences) in 224 minutes, with the learning rate warmed up over 100 steps and then decayed to zero. The training loss fell from about 0.49 over the first 500 steps to about 0.105 by mid-epoch and about 0.077 over the last 500 steps, so it was still improving slowly at the end. Only the LoRA adapter, the classification head and the tokenizer are saved, a few tens of MB, since the base model is downloaded again from HuggingFace when scoring.